# Tutorial 12 — Agentic RAG: Multi-Tool Scientific Research Agent
**Author:** Himanshu Goel | [Website](https://hgoelgithub.github.io)

**Agentic RAG** goes far beyond vanilla RAG by giving an LLM a *toolbelt* and letting it decide:
- **Which sources to query** — PubMed, arXiv, Semantic Scholar, or the open web
- **Which papers to fully index** — fetch & chunk only the most promising results
- **When to search vs. synthesize** — retrieves evidence iteratively before writing an answer

### Available tools
| Tool | Source | Purpose |
|------|--------|---------|
| `pubmed_search` | NCBI E-utilities | Discover peer-reviewed papers by query |
| `fetch_and_index_paper` | PMC full-text XML | Fetch a paper by PMID and index into vector DB |
| `arxiv_search` | arXiv API | Find preprints and ML/CS methods |
| `semantic_scholar_search` | Semantic Scholar API | Cross-domain search + citation counts |
| `web_search` | DuckDuckGo | General web results — docs, news, blogs |
| `fetch_url` | Any URL | Parse & index any webpage |
| `vector_search` | ChromaDB | Semantic search over all indexed content |
| `crossref_lookup` | CrossRef API | DOI → journal / author / citation metadata |

### Vanilla RAG vs Agentic RAG
| Aspect | Vanilla RAG | Agentic RAG |
|--------|-------------|-------------|
| Knowledge source | Fixed pre-indexed corpus | Dynamically searched at query time |
| Retrieval strategy | Single one-shot lookup | Multi-step, tool-guided |
| Coverage | Pre-indexed documents only | Live internet + multiple databases |
| Multi-hop reasoning | Limited | Yes — agent chains tool calls |
| Source diversity | One database | PubMed + arXiv + Semantic Scholar + Web |

In [ ]:
!pip install openai chromadb sentence-transformers -q
!pip install duckduckgo-search beautifulsoup4 requests python-dotenv -q
!pip install 'transformers>=4.41.0,<5.0' -q

import transformers; print('transformers', transformers.__version__)
import openai;       print('openai      ', openai.__version__)

In [ ]:
!pip install pdfplumber python-pptx pillow -q
!pip install networkx matplotlib pyvis -q
!pip install gradio -q

import networkx;   print('networkx   ', networkx.__version__)
import pdfplumber; print('pdfplumber ', pdfplumber.__version__)
import gradio;     print('gradio     ', gradio.__version__)

In [3]:
import os, json, re, time, hashlib
import requests
import xml.etree.ElementTree as ET
import urllib.parse

import chromadb
from openai import OpenAI
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv

load_dotenv()  # reads OPENAI_API_KEY (and optional NCBI_API_KEY) from .env

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY', '')
NCBI_API_KEY   = os.getenv('NCBI_API_KEY', '')  # optional: 10 req/s vs 3 req/s

# ── Embedding model (local, no API key needed) ────────────────────────────────
# all-MiniLM-L6-v2: 22M params, fast, good enough for retrieval
print('Loading embedding model...')
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
print('Embedding model ready.')

# ── In-memory ChromaDB vector store ──────────────────────────────────────────
# EphemeralClient avoids SQLite file-lock issues when re-running cells
chroma_client = chromadb.EphemeralClient()
collection    = chroma_client.get_or_create_collection(
    name='agentic_rag',
    metadata={'hnsw:space': 'cosine'}
)

# ── OpenAI client ─────────────────────────────────────────────────────────────
client = OpenAI(api_key=OPENAI_API_KEY)

# ── Agent state ───────────────────────────────────────────────────────────────
_indexed_ids: set[str] = set()  # track already-indexed doc IDs (idempotency)

print(f'Vector DB ready. OpenAI client ready.')
print(f'NCBI API key: {"set" if NCBI_API_KEY else "not set (3 req/s limit applies)"}')

Loading embedding model...
Embedding model ready.
Vector DB ready. OpenAI client ready.
NCBI API key: not set (3 req/s limit applies)


In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter
import pathlib, base64

KG = nx.DiGraph()

KG_TYPE_COLORS = {
    'Method':  '#4CAF50', 'Drug':    '#2196F3', 'Protein': '#FF9800',
    'Paper':   '#9C27B0', 'Author':  '#F44336', 'Dataset': '#00BCD4',
    'Concept': '#795548', 'Disease': '#E91E63', 'Unknown': '#9E9E9E',
}

print('Knowledge graph ready.')

## Step 1 — Define the Tools

Each tool is a plain Python function. The agent will choose which ones to call and with what arguments. Results flow back into the conversation as `tool_result` messages.

In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# Tool 1: pubmed_search
#
# Queries NCBI PubMed via the E-utilities esearch + efetch API.
# Returns lightweight metadata (title, abstract snippet, PMID, PMC ID, DOI)
# so the agent can decide which papers are worth fully indexing.
# ─────────────────────────────────────────────────────────────────────────────
def tool_pubmed_search(query: str, max_results: int = 10) -> dict:
    base   = 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/'
    params = {'db': 'pubmed', 'term': query, 'retmax': max_results, 'retmode': 'json'}
    if NCBI_API_KEY:
        params['api_key'] = NCBI_API_KEY

    try:
        r    = requests.get(f'{base}esearch.fcgi', params=params, timeout=15)
        pmids = r.json()['esearchresult']['idlist']
    except Exception as e:
        return {'error': str(e), 'results': []}

    if not pmids:
        return {'results': [], 'count': 0}

    fetch = requests.get(
        f'{base}efetch.fcgi',
        params={'db': 'pubmed', 'id': ','.join(pmids), 'rettype': 'abstract', 'retmode': 'xml'},
        timeout=20
    )
    root    = ET.fromstring(fetch.text)
    results = []
    for art in root.iter('PubmedArticle'):
        pmid  = art.findtext('.//PMID', '')
        title = ''.join(''.join(t.itertext()) for t in art.iter('ArticleTitle'))
        abstr = ''.join(''.join(t.itertext()) for t in art.iter('AbstractText'))[:400]
        year  = art.findtext('.//PubDate/Year', '?')
        pmcid = art.findtext(".//ArticleId[@IdType='pmc']", '')
        doi   = art.findtext(".//ArticleId[@IdType='doi']", '')
        results.append({
            'pmid': pmid, 'title': title[:120], 'abstract_snippet': abstr,
            'year': year, 'pmcid': pmcid, 'doi': doi,
            'has_fulltext': bool(pmcid)
        })

    return {'results': results, 'count': len(results)}


# Quick smoke test
demo = tool_pubmed_search('SILCS binding affinity drug discovery', max_results=3)
print(f"PubMed returned {demo['count']} results")
for r in demo['results']:
    print(f"  [{r['year']}] PMID {r['pmid']} | full-text={r['has_fulltext']} | {r['title'][:70]}...")

PubMed returned 3 results
  [2023] PMID 37218059 | full-text=True | Ranking mAb-excipient interactions in biologics formulations by NMR sp...
  [2022] PMID 35913731 | full-text=True | SILCS-RNA: Toward a Structure-Based Drug Design Approach for Targeting...
  [2022] PMID 35210743 | full-text=True | Application of Site-Identification by Ligand Competitive Saturation in...


In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# Tool 2: fetch_and_index_paper
#
# Given a PMID:
#   1. Fetches the full article XML from PubMed Central if it's open-access
#   2. Falls back to the abstract if no full text is available
#   3. Splits into 700-char chunks (100-char overlap)
#   4. Embeds and inserts every chunk into ChromaDB
#
# Re-indexing the same PMID is a no-op (idempotent).
# ─────────────────────────────────────────────────────────────────────────────
def tool_fetch_and_index_paper(pmid: str) -> dict:
    if pmid in _indexed_ids:
        return {'status': 'already_indexed', 'pmid': pmid}

    base = 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/'
    try:
        fetch = requests.get(
            f'{base}efetch.fcgi',
            params={'db': 'pubmed', 'id': pmid, 'rettype': 'abstract', 'retmode': 'xml'},
            timeout=20
        )
        root = ET.fromstring(fetch.text)
    except Exception as e:
        return {'status': 'error', 'pmid': pmid, 'error': str(e)}

    art = root.find('.//PubmedArticle')
    if art is None:
        return {'status': 'not_found', 'pmid': pmid}

    title = ''.join(''.join(t.itertext()) for t in art.iter('ArticleTitle'))
    abstr = ''.join(''.join(t.itertext()) for t in art.iter('AbstractText'))
    year  = art.findtext('.//PubDate/Year', '?')
    pmcid = art.findtext(".//ArticleId[@IdType='pmc']", '')
    doi   = art.findtext(".//ArticleId[@IdType='doi']", '')

    content = None
    source  = 'abstract'

    if pmcid:
        try:
            pmc      = requests.get(
                f'{base}efetch.fcgi',
                params={'db': 'pmc', 'id': pmcid, 'rettype': 'xml', 'retmode': 'xml'},
                timeout=25
            )
            pmc_root = ET.fromstring(pmc.text)
            sections = []
            for sec in pmc_root.iter('sec'):
                sec_title = sec.findtext('title', '').strip()
                sec_text  = ' '.join(''.join(p.itertext()) for p in sec.iter('p')).strip()
                if sec_text:
                    sections.append(f'[{sec_title or "Section"}]\n{sec_text}')
            if sections:
                content = f'Title: {title}\n\n' + '\n\n'.join(sections)
                source  = 'fulltext'
            time.sleep(0.35)   # NCBI rate limit: stay under 3 req/s unauthenticated
        except Exception:
            pass

    if content is None:
        if abstr.strip():
            content = f'Title: {title}\n\nAbstract: {abstr}'
        else:
            return {'status': 'no_content', 'pmid': pmid}

    # ── Chunk into overlapping segments ──────────────────────────────────────
    chunk_size, overlap = 700, 100
    chunks = []
    for i in range(0, len(content), chunk_size - overlap):
        chunk = content[i: i + chunk_size].strip()
        if chunk:
            chunks.append(chunk)

    # ── Embed and upsert into ChromaDB ────────────────────────────────────────
    embeddings = embedder.encode(chunks).tolist()
    ids        = [f'pmid_{pmid}_{j}' for j in range(len(chunks))]
    metas      = [{'pmid': pmid, 'pmcid': pmcid, 'title': title[:100],
                   'year': year, 'source': source, 'doi': doi}
                  for _ in chunks]
    collection.upsert(documents=chunks, embeddings=embeddings, ids=ids, metadatas=metas)

    _indexed_ids.add(pmid)
    return {
        'status': 'indexed', 'pmid': pmid, 'title': title[:100],
        'source': source, 'chunks_added': len(chunks),
        'total_in_db': collection.count()
    }


# Test: fetch and index the first result from the previous search
if demo['results']:
    test_pmid = demo['results'][0]['pmid']
    result    = tool_fetch_and_index_paper(test_pmid)
    print(result)

{'status': 'indexed', 'pmid': '37218059', 'title': 'Ranking mAb-excipient interactions in biologics formulations by NMR spectroscopy and computational a', 'source': 'fulltext', 'chunks_added': 153, 'total_in_db': 153}


In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# Tool 3: arxiv_search
#
# Queries the arXiv Atom feed API — free, no key required.
# Great for finding recent ML/CS preprints that haven't hit PubMed yet.
# Automatically embeds and indexes each abstract into ChromaDB.
# ─────────────────────────────────────────────────────────────────────────────
def tool_arxiv_search(query: str, max_results: int = 5) -> dict:
    url = (
        f'https://export.arxiv.org/api/query'
        f'?search_query=all:{urllib.parse.quote(query)}'
        f'&max_results={max_results}&sortBy=relevance'
    )
    try:
        r    = requests.get(url, timeout=15)
        root = ET.fromstring(r.text)
    except Exception as e:
        return {'error': str(e), 'results': []}

    ns      = {'atom': 'http://www.w3.org/2005/Atom'}
    results = []
    for entry in root.findall('atom:entry', ns):
        arxiv_id  = entry.findtext('atom:id', '', ns).split('/abs/')[-1]
        title     = entry.findtext('atom:title', '', ns).strip().replace('\n', ' ')
        summary   = entry.findtext('atom:summary', '', ns).strip().replace('\n', ' ')
        published = entry.findtext('atom:published', '', ns)[:10]
        authors   = [a.findtext('atom:name', '', ns)
                     for a in entry.findall('atom:author', ns)][:4]

        # Index abstract in ChromaDB
        doc_id  = f'arxiv_{arxiv_id.replace("/", "_")}'
        content = f'Title: {title}\n\nAbstract: {summary}'
        if doc_id not in _indexed_ids:
            emb = embedder.encode([content]).tolist()
            collection.upsert(
                documents=[content], embeddings=emb, ids=[doc_id],
                metadatas=[{'arxiv_id': arxiv_id, 'title': title[:100],
                            'year': published[:4], 'source': 'arxiv',
                            'pmid': '', 'pmcid': ''}]
            )
            _indexed_ids.add(doc_id)

        results.append({
            'arxiv_id': arxiv_id, 'title': title[:120],
            'abstract_snippet': summary[:300], 'published': published,
            'authors': authors
        })

    return {'results': results, 'count': len(results), 'indexed': len(results)}


demo_arxiv = tool_arxiv_search('protein ligand binding deep learning', max_results=3)
def tool_arxiv_search(query: str, max_results: int = 5) -> dict:
    url = (
        f'https://export.arxiv.org/api/query'
        f'?search_query=all:{urllib.parse.quote(query)}'
        f'&max_results={max_results}&sortBy=relevance'
    )
    try:
        r    = requests.get(url, timeout=15)
        root = ET.fromstring(r.text)
    except Exception as e:
        return {'error': str(e), 'results': [], 'count': 0}

    ns      = {'atom': 'http://www.w3.org/2005/Atom'}
    results = []
    for entry in root.findall('atom:entry', ns):
        arxiv_id  = entry.findtext('atom:id', '', ns).split('/abs/')[-1]
        title     = entry.findtext('atom:title', '', ns).strip().replace('\n', ' ')
        summary   = entry.findtext('atom:summary', '', ns).strip().replace('\n', ' ')
        published = entry.findtext('atom:published', '', ns)[:10]
        authors   = [a.findtext('atom:name', '', ns)
                     for a in entry.findall('atom:author', ns)][:4]

        # Index abstract in ChromaDB
        doc_id  = f'arxiv_{arxiv_id.replace("/", "_")}'
        content = f'Title: {title}\n\nAbstract: {summary}'
        if doc_id not in _indexed_ids:
            emb = embedder.encode([content]).tolist()
            collection.upsert(
                documents=[content], embeddings=emb, ids=[doc_id],
                metadatas=[{'arxiv_id': arxiv_id, 'title': title[:100],
                            'year': published[:4], 'source': 'arxiv',
                            'pmid': '', 'pmcid': ''}]
            )
            _indexed_ids.add(doc_id)

        results.append({
            'arxiv_id': arxiv_id, 'title': title[:120],
            'abstract_snippet': summary[:300], 'published': published,
            'authors': authors
        })

    return {'results': results, 'count': len(results), 'indexed': len(results)}


demo_arxiv = tool_arxiv_search('protein ligand binding deep learning', max_results=3)
print(f"arXiv returned {demo_arxiv.get('count', 0)} results")
for r in demo_arxiv['results']:
    print(f"  [{r['published']}] arXiv:{r['arxiv_id']} | {r['title'][:70]}...")
for r in demo_arxiv['results']:
    print(f"  [{r['published']}] arXiv:{r['arxiv_id']} | {r['title'][:70]}...")

arXiv returned 3 results
  [2024-05-23] arXiv:2405.14108v7 | Assessing the potential of deep learning for protein-ligand docking...
  [2017-11-22] arXiv:1711.10540v2 | ISLAND: In-Silico Prediction of Proteins Binding Affinity Using Sequen...
  [2023-06-19] arXiv:2306.11113v2 | Learn to Accumulate Evidence from All Training Samples: Theory and Pra...
  [2024-05-23] arXiv:2405.14108v7 | Assessing the potential of deep learning for protein-ligand docking...
  [2017-11-22] arXiv:1711.10540v2 | ISLAND: In-Silico Prediction of Proteins Binding Affinity Using Sequen...
  [2023-06-19] arXiv:2306.11113v2 | Learn to Accumulate Evidence from All Training Samples: Theory and Pra...


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Tool 4: semantic_scholar_search
#
# Queries the Semantic Scholar Graph API (free, no key required).
# Returns citation counts — useful for identifying influential papers.
# Cross-domain: covers computer science, medicine, biology, physics, etc.
# ─────────────────────────────────────────────────────────────────────────────
def tool_semantic_scholar_search(query: str, max_results: int = 5) -> dict:
    try:
        r = requests.get(
            'https://api.semanticscholar.org/graph/v1/paper/search',
            params={
                'query': query, 'limit': max_results,
                'fields': 'title,abstract,year,authors,citationCount,externalIds'
            },
            headers={'User-Agent': 'AgenticRAG/1.0 (educational use)'},
            timeout=15
        )
    except Exception as e:
        return {'error': str(e), 'results': [], 'count': 0}

    if r.status_code != 200:
        return {'error': f'HTTP {r.status_code}', 'results': [], 'count': 0}

    results = []
    for p in r.json().get('data', []):
        paper_id = p.get('paperId', '')
        title    = p.get('title', '')
        abstract = (p.get('abstract') or '')[:400]
        year     = p.get('year', '?')
        cites    = p.get('citationCount', 0)
        pmid     = p.get('externalIds', {}).get('PubMed', '')

        if abstract:
            doc_id  = f'ss_{paper_id}'
            content = f'Title: {title}\n\nAbstract: {abstract}'
            if doc_id not in _indexed_ids:
                emb = embedder.encode([content]).tolist()
                collection.upsert(
                    documents=[content], embeddings=emb, ids=[doc_id],
                    metadatas=[{'ss_id': paper_id, 'title': title[:100],
                                'year': str(year), 'source': 'semantic_scholar',
                                'pmid': pmid, 'pmcid': '', 'citations': str(cites)}]
                )
                _indexed_ids.add(doc_id)

        results.append({
            'paper_id': paper_id, 'title': title[:120],
            'abstract_snippet': abstract[:200], 'year': year,
            'citations': cites, 'pmid': pmid
        })

    return {'results': results, 'count': len(results)}


demo_s2 = tool_semantic_scholar_search('SILCS fragment binding free energy', max_results=3)
print(f"Semantic Scholar returned {demo_s2['count']} results")
for r in demo_s2['results']:
    print(f"  [{r['year']}] {r['citations']} citations | {r['title'][:70]}...")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Tool 5: web_search
#
# DuckDuckGo search — no API key required.
# Returns up to max_results snippets with URL and title.
# Use for documentation, news, blog posts, or any non-academic query.
# ─────────────────────────────────────────────────────────────────────────────
def tool_web_search(query: str, max_results: int = 5) -> dict:
    try:
        from duckduckgo_search import DDGS
        with DDGS() as ddgs:
            raw = list(ddgs.text(query, max_results=max_results))
        return {
            'results': [
                {'title': r.get('title', ''),
                 'url':   r.get('href', ''),
                 'snippet': r.get('body', '')[:300]}
                for r in raw
            ],
            'count': len(raw)
        }
    except Exception as e:
        return {'error': str(e), 'results': []}


# ─────────────────────────────────────────────────────────────────────────────
# Tool 6: fetch_url
#
# Fetches a URL, strips HTML tags with BeautifulSoup, and indexes the
# extracted text into ChromaDB for downstream vector_search queries.
# Useful when the agent finds a relevant page via web_search.
# ─────────────────────────────────────────────────────────────────────────────
def tool_fetch_url(url: str, max_chars: int = 3000) -> dict:
    try:
        from bs4 import BeautifulSoup
        headers = {'User-Agent': 'Mozilla/5.0 (compatible; AgenticRAG/1.0)'}
        r       = requests.get(url, headers=headers, timeout=20)
        soup    = BeautifulSoup(r.text, 'html.parser')
        for tag in soup(['script', 'style', 'nav', 'footer', 'header', 'aside']):
            tag.decompose()
        text = soup.get_text(separator='\n', strip=True)
        text = re.sub(r'\n{3,}', '\n\n', text)[:max_chars]

        # get_text() handles nested tags safely; .string returns None for multi-child elements
        page_title = soup.title.get_text(strip=True) if soup.title else url[:80]

        # hashlib.md5 is stable across restarts; hash() uses a random seed per process
        doc_id = 'web_' + hashlib.md5(url.encode()).hexdigest()[:12]
        emb    = embedder.encode([text]).tolist()
        collection.upsert(
            documents=[text], embeddings=emb, ids=[doc_id],
            metadatas=[{'url': url, 'source': 'web', 'title': page_title[:100],
                        'pmid': '', 'pmcid': '', 'year': '?'}]
        )
        _indexed_ids.add(doc_id)

        return {'url': url, 'title': page_title, 'content_preview': text[:500],
                'chars_indexed': len(text), 'total_in_db': collection.count()}
    except Exception as e:
        return {'error': str(e), 'url': url}


print('web_search and fetch_url tools defined.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Tool 9: load_local_file
#
# Loads any local document and indexes its content into ChromaDB.
# Supported formats:
#   .pdf        — pdfplumber extracts text page-by-page (tables included)
#   .pptx/.ppt  — python-pptx extracts slide text + speaker notes
#   .html/.htm  — BeautifulSoup parses local HTML files
#   .png/.jpg/.jpeg/.webp/.gif — GPT-4o vision describes the image content
#
# All extracted text is chunked and upserted into ChromaDB exactly like
# web pages or PubMed abstracts — vector_search queries work across them.
# ─────────────────────────────────────────────────────────────────────────────
_IMAGE_MIME = {
    'png':  'image/png', 'jpg':  'image/jpeg', 'jpeg': 'image/jpeg',
    'webp': 'image/webp', 'gif': 'image/gif',  'bmp':  'image/bmp',
}

def tool_load_local_file(file_path: str) -> dict:
    path = pathlib.Path(file_path).expanduser().resolve()
    if not path.exists():
        return {'error': f'File not found: {file_path}'}

    ext          = path.suffix.lower().lstrip('.')
    content      = ''
    source_label = ext

    # ── PDF ──────────────────────────────────────────────────────────────────
    if ext == 'pdf':
        import pdfplumber
        pages = []
        with pdfplumber.open(str(path)) as pdf:
            for i, page in enumerate(pdf.pages):
                text = page.extract_text() or ''
                # Also try to extract tables as plain text rows
                for table in page.extract_tables():
                    rows = [' | '.join(str(c) for c in row if c) for row in table if row]
                    text += '\n' + '\n'.join(rows)
                if text.strip():
                    pages.append(f'[Page {i+1}]\n{text.strip()}')
        content = '\n\n'.join(pages)

    # ── PowerPoint ───────────────────────────────────────────────────────────
    elif ext in ('pptx', 'ppt'):
        from pptx import Presentation
        prs    = Presentation(str(path))
        slides = []
        for i, slide in enumerate(prs.slides):
            parts = []
            for shape in slide.shapes:
                if shape.has_text_frame:
                    parts.append(shape.text_frame.text.strip())
            # Include speaker notes if present
            if slide.has_notes_slide:
                notes = slide.notes_slide.notes_text_frame.text.strip()
                if notes:
                    parts.append(f'[Notes] {notes}')
            if parts:
                slides.append(f'[Slide {i+1}]\n' + '\n'.join(p for p in parts if p))
        content = '\n\n'.join(slides)

    # ── Local HTML ───────────────────────────────────────────────────────────
    elif ext in ('html', 'htm'):
        from bs4 import BeautifulSoup
        raw  = path.read_text(encoding='utf-8', errors='ignore')
        soup = BeautifulSoup(raw, 'html.parser')
        for tag in soup(['script', 'style', 'nav', 'footer', 'header', 'aside']):
            tag.decompose()
        content = soup.get_text(separator='\n', strip=True)
        content = re.sub(r'\n{3,}', '\n\n', content)
        source_label = 'html_local'

    # ── Image (GPT-4o vision) ─────────────────────────────────────────────────
    elif ext in _IMAGE_MIME:
        mime    = _IMAGE_MIME[ext]
        img_b64 = base64.b64encode(path.read_bytes()).decode()
        resp    = client.chat.completions.create(
            model='gpt-4o',
            max_tokens=1024,
            messages=[{
                'role': 'user',
                'content': [
                    {
                        'type': 'image_url',
                        'image_url': {'url': f'data:{mime};base64,{img_b64}', 'detail': 'high'}
                    },
                    {
                        'type': 'text',
                        'text': (
                            'Describe this image in full scientific detail. '
                            'Extract every piece of text, all axis labels, figure captions, '
                            'table contents, equations, and numerical values visible. '
                            'Explain what the figure or diagram shows scientifically.'
                        )
                    }
                ]
            }]
        )
        content      = f'[Image: {path.name}]\n\n' + resp.choices[0].message.content
        source_label = 'image'

    else:
        supported = 'pdf, pptx, html, ' + ', '.join(_IMAGE_MIME.keys())
        return {'error': f'Unsupported extension ".{ext}". Supported: {supported}'}

    if not content.strip():
        return {'error': 'No text content extracted', 'file': str(path)}

    # ── Chunk and index into ChromaDB ─────────────────────────────────────────
    chunk_size, overlap = 700, 100
    chunks = []
    for i in range(0, len(content), chunk_size - overlap):
        chunk = content[i: i + chunk_size].strip()
        if chunk:
            chunks.append(chunk)

    file_id    = 'file_' + hashlib.md5(str(path).encode()).hexdigest()[:12]
    embeddings = embedder.encode(chunks).tolist()
    ids        = [f'{file_id}_{j}' for j in range(len(chunks))]
    metas      = [{'source': source_label, 'title': path.stem, 'file': path.name,
                   'pmid': '', 'pmcid': '', 'year': '?'}
                  for _ in chunks]
    collection.upsert(documents=chunks, embeddings=embeddings, ids=ids, metadatas=metas)
    _indexed_ids.add(file_id)

    return {
        'file':          path.name,
        'type':          source_label,
        'pages_slides':  content.count('[Page ') + content.count('[Slide '),
        'chars':         len(content),
        'chunks_indexed': len(chunks),
        'total_in_db':   collection.count(),
        'preview':       content[:300],
    }


print('tool_load_local_file ready — supports: pdf, pptx, html, png/jpg/webp (vision)')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Tool 10: kg_extract
#
# Takes a block of text (e.g. from an indexed paper) and calls GPT-4o-mini
# to extract named entities and typed relationships, then stores them in the
# global NetworkX KG.
#
# Entity types: Method, Drug, Protein, Paper, Author, Dataset, Concept, Disease
# Relation examples: "uses", "proposes", "evaluates_on", "compares",
#                    "treats", "inhibits", "cites", "outperforms"
# ─────────────────────────────────────────────────────────────────────────────
def tool_kg_extract(text: str, source_id: str = '') -> dict:
    prompt = (
        'Extract scientific entities and relationships from the text below.\n'
        'Return a JSON object with exactly two keys:\n'
        '  "entities": list of {"name": str, "type": "Method|Drug|Protein|Paper|Author|Dataset|Concept|Disease"}\n'
        '  "relations": list of {"from": str, "relation": str, "to": str}\n'
        'Include only clearly stated facts. Be concise.\n\n'
        f'Text:\n{text[:2500]}'
    )
    try:
        resp = client.chat.completions.create(
            model='gpt-4o-mini',
            messages=[{'role': 'user', 'content': prompt}],
            response_format={'type': 'json_object'},
            max_tokens=1024,
        )
        data = json.loads(resp.choices[0].message.content)
    except Exception as e:
        return {'error': str(e)}

    nodes_added = 0
    edges_added = 0

    for ent in data.get('entities', []):
        name = str(ent.get('name', '')).strip()
        if name:
            KG.add_node(name,
                        type=ent.get('type', 'Unknown'),
                        source=source_id)
            nodes_added += 1

    for rel in data.get('relations', []):
        frm = str(rel.get('from', '')).strip()
        to  = str(rel.get('to',   '')).strip()
        tag = str(rel.get('relation', '')).strip()
        if frm and to:
            KG.add_edge(frm, to, relation=tag, source=source_id)
            edges_added += 1

    return {
        'nodes_added':  nodes_added,
        'edges_added':  edges_added,
        'total_nodes':  KG.number_of_nodes(),
        'total_edges':  KG.number_of_edges(),
        'entity_types': dict(Counter(d.get('type', 'Unknown')
                                     for _, d in KG.nodes(data=True))),
    }


# ─────────────────────────────────────────────────────────────────────────────
# Tool 11: kg_search
#
# Given a starting entity name, returns all nodes and edges reachable within
# `depth` hops in the knowledge graph.  Useful for answering "what is related
# to X?" queries and for the agent to plan follow-up retrievals.
# ─────────────────────────────────────────────────────────────────────────────
def tool_kg_search(entity: str, depth: int = 2) -> dict:
    if KG.number_of_nodes() == 0:
        return {'error': 'Knowledge graph is empty. Call kg_extract first.'}

    # Case-insensitive fuzzy node match
    matches = [n for n in KG.nodes() if entity.lower() in n.lower()]
    if not matches:
        return {
            'error':           f'"{entity}" not found in KG.',
            'available_nodes': sorted(KG.nodes())[:30],
        }

    center = matches[0]  # best match
    reachable = nx.single_source_shortest_path_length(KG, center, cutoff=depth)
    sub = KG.subgraph(list(reachable.keys()))

    edges_info = [
        {'from': u, 'relation': d.get('relation', ''), 'to': v, 'hops': reachable.get(u, 0)}
        for u, v, d in sub.edges(data=True)
    ]

    # Also get incoming edges to the center
    in_edges = [
        {'from': u, 'relation': d.get('relation', ''), 'to': center}
        for u, _, d in KG.in_edges(center, data=True)
        if u not in reachable
    ]

    return {
        'center':     center,
        'nodes':      list(reachable.keys()),
        'out_edges':  edges_info[:40],
        'in_edges':   in_edges[:10],
        'node_count': len(reachable),
    }


# ─────────────────────────────────────────────────────────────────────────────
# Tool 12: kg_visualize
#
# Renders the full knowledge graph inline in the notebook using two backends:
#   • Matplotlib — static PNG, colour-coded by entity type, edge labels
#   • PyVis      — interactive HTML graph (drag, zoom, hover) saved to kg.html
#
# Returns a summary of node/edge counts and type breakdown.
# ─────────────────────────────────────────────────────────────────────────────
def tool_kg_visualize(backend: str = 'matplotlib') -> dict:
    if KG.number_of_nodes() == 0:
        return {'error': 'Knowledge graph is empty. Call kg_extract first.'}

    type_counts = dict(Counter(d.get('type', 'Unknown')
                               for _, d in KG.nodes(data=True)))

    if backend == 'pyvis':
        from pyvis.network import Network
        from IPython.display import display, HTML

        net = Network(height='600px', width='100%', directed=True, notebook=True,
                      bgcolor='#1a1a2e', font_color='white')
        net.barnes_hut(spring_length=120)

        for node, data in KG.nodes(data=True):
            color = KG_TYPE_COLORS.get(data.get('type', 'Unknown'), '#9E9E9E')
            net.add_node(node, label=node, color=color,
                         title=f"{data.get('type','?')} | source: {data.get('source','')}",
                         size=20)

        for u, v, data in KG.edges(data=True):
            net.add_edge(u, v, label=data.get('relation', ''), color='#aaaaaa')

        net.save_graph('kg.html')
        display(HTML('kg.html'))

    else:  # matplotlib (default)
        fig, ax = plt.subplots(figsize=(16, 11))
        pos = nx.spring_layout(KG, k=2.5, seed=42)

        node_colors = [KG_TYPE_COLORS.get(KG.nodes[n].get('type', 'Unknown'), '#9E9E9E')
                       for n in KG.nodes()]
        node_sizes  = [600 + 40 * KG.degree(n) for n in KG.nodes()]

        nx.draw_networkx_nodes(KG, pos, node_color=node_colors,
                               node_size=node_sizes, alpha=0.92, ax=ax)
        nx.draw_networkx_labels(KG, pos, font_size=7,
                                font_weight='bold', ax=ax)
        nx.draw_networkx_edges(KG, pos, edge_color='#888888',
                               arrows=True, arrowsize=14,
                               connectionstyle='arc3,rad=0.1', ax=ax)
        edge_labels = {(u, v): d.get('relation', '')
                       for u, v, d in KG.edges(data=True)}
        nx.draw_networkx_edge_labels(KG, pos, edge_labels,
                                     font_size=6, ax=ax)

        legend = [mpatches.Patch(color=c, label=t)
                  for t, c in KG_TYPE_COLORS.items()
                  if t in type_counts]
        ax.legend(handles=legend, loc='upper left', fontsize=8)
        ax.set_title(
            f'Knowledge Graph — {KG.number_of_nodes()} nodes, '
            f'{KG.number_of_edges()} edges',
            fontsize=13
        )
        ax.axis('off')
        plt.tight_layout()
        plt.show()

    return {
        'nodes':       KG.number_of_nodes(),
        'edges':       KG.number_of_edges(),
        'entity_types': type_counts,
    }


print('KG tools ready: kg_extract | kg_search | kg_visualize')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Tool 7: vector_search
#
# Semantic search over all content currently in ChromaDB.
# The agent should call this AFTER indexing papers to extract specific passages
# before writing its final answer.
#
# Optional filter_source limits results to a single origin:
#   'fulltext' | 'abstract' | 'arxiv' | 'semantic_scholar' | 'web'
# ─────────────────────────────────────────────────────────────────────────────
def tool_vector_search(query: str, k: int = 8, filter_source: str = '') -> dict:
    count = collection.count()
    if count == 0:
        return {
            'results': [],
            'message': 'Vector DB is empty. Use pubmed_search / arxiv_search tools first.'
        }

    emb   = embedder.encode([query]).tolist()
    where = {'source': {'$eq': filter_source}} if filter_source else None

    res = collection.query(
        query_embeddings=emb,
        n_results=min(k, count),
        where=where,
        include=['documents', 'metadatas', 'distances']
    )

    hits = []
    for doc, meta, dist in zip(
        res['documents'][0],
        res['metadatas'][0],
        res['distances'][0]
    ):
        hits.append({
            'score':    round(1 - dist, 4),
            'content':  doc[:500],
            'metadata': {key: val for key, val in meta.items() if val}  # renamed k→key to avoid shadowing param k
        })

    return {'results': hits, 'count': len(hits), 'total_indexed': count}


# ─────────────────────────────────────────────────────────────────────────────
# Tool 8: crossref_lookup
#
# Given a DOI, fetches rich metadata from the CrossRef REST API:
# journal name, author list, publication year, citation count, and abstract.
# Useful for verifying impact and provenance of a specific paper.
# ─────────────────────────────────────────────────────────────────────────────
def tool_crossref_lookup(doi: str) -> dict:
    try:
        r = requests.get(
            f'https://api.crossref.org/works/{urllib.parse.quote(doi, safe="")}',
            headers={'User-Agent': 'AgenticRAG/1.0 (educational use)'},
            timeout=12
        )
        if r.status_code != 200:
            return {'error': f'CrossRef returned HTTP {r.status_code}', 'doi': doi}
        m = r.json()['message']
        return {
            'doi': doi,
            'title': ' '.join(m.get('title', [])),
            'journal': (m.get('container-title') or ['?'])[0],
            'year': (m.get('published', {}).get('date-parts') or [[None]])[0][0],
            'authors': [
                f"{a.get('given', '')} {a.get('family', '')}".strip()
                for a in m.get('author', [])[:5]
            ],
            'cited_by': m.get('is-referenced-by-count', 0),
            'abstract': (m.get('abstract') or '')[:400]
        }
    except Exception as e:
        return {'error': str(e), 'doi': doi}


print('vector_search and crossref_lookup tools defined.')
print(f'Vector DB currently holds {collection.count()} chunks.')

## Step 2 — Tool Registry & Schemas

The model receives tools as JSON schemas. Descriptions drive *when* the model calls each tool.

In [ ]:
# ── Tool registry: maps tool name → Python callable ──────────────────────────
TOOL_REGISTRY = {
    'pubmed_search':           tool_pubmed_search,
    'fetch_and_index_paper':   tool_fetch_and_index_paper,
    'arxiv_search':            tool_arxiv_search,
    'semantic_scholar_search': tool_semantic_scholar_search,
    'web_search':              tool_web_search,
    'fetch_url':               tool_fetch_url,
    'vector_search':           tool_vector_search,
    'crossref_lookup':         tool_crossref_lookup,
    # ── New in this tutorial ──────────────────────────────────────────────────
    'load_local_file':         tool_load_local_file,
    'kg_extract':              tool_kg_extract,
    'kg_search':               tool_kg_search,
    'kg_visualize':            tool_kg_visualize,
}

# ── OpenAI tool schemas ───────────────────────────────────────────────────────
TOOLS = [
    {
        'type': 'function',
        'function': {
            'name': 'pubmed_search',
            'description': (
                'Search PubMed for peer-reviewed scientific papers. Returns title, '
                'abstract snippet, year, PMID, DOI, and whether full text is available. '
                'Use this first to discover relevant papers. Supports Boolean operators '
                'and field tags: [ti] title, [ab] abstract, [au] author.'
            ),
            'parameters': {
                'type': 'object',
                'properties': {
                    'query':       {'type': 'string',  'description': 'PubMed search query'},
                    'max_results': {'type': 'integer', 'description': 'Papers to return (default 10)', 'default': 10}
                },
                'required': ['query']
            }
        }
    },
    {
        'type': 'function',
        'function': {
            'name': 'fetch_and_index_paper',
            'description': (
                'Fetch full text (or abstract fallback) for a PubMed PMID and index '
                'it in the vector DB. Call this for the most relevant PMIDs discovered '
                'via pubmed_search before running vector_search.'
            ),
            'parameters': {
                'type': 'object',
                'properties': {
                    'pmid': {'type': 'string', 'description': 'PubMed ID to fetch and index'}
                },
                'required': ['pmid']
            }
        }
    },
    {
        'type': 'function',
        'function': {
            'name': 'arxiv_search',
            'description': (
                'Search arXiv for preprints. Best for ML/AI methods and recent work '
                'not yet on PubMed. Auto-indexes all abstracts into the vector DB.'
            ),
            'parameters': {
                'type': 'object',
                'properties': {
                    'query':       {'type': 'string',  'description': 'arXiv search query'},
                    'max_results': {'type': 'integer', 'description': 'Papers to return (default 5)', 'default': 5}
                },
                'required': ['query']
            }
        }
    },
    {
        'type': 'function',
        'function': {
            'name': 'semantic_scholar_search',
            'description': (
                'Search Semantic Scholar — cross-domain, includes citation counts. '
                'Good complement to PubMed. Auto-indexes all results with abstracts.'
            ),
            'parameters': {
                'type': 'object',
                'properties': {
                    'query':       {'type': 'string',  'description': 'Search query'},
                    'max_results': {'type': 'integer', 'description': 'Papers to return (default 5)', 'default': 5}
                },
                'required': ['query']
            }
        }
    },
    {
        'type': 'function',
        'function': {
            'name': 'web_search',
            'description': (
                'Search the web with DuckDuckGo. No API key required. '
                'Use for documentation, news, or when academic databases come up empty.'
            ),
            'parameters': {
                'type': 'object',
                'properties': {
                    'query':       {'type': 'string',  'description': 'Web search query'},
                    'max_results': {'type': 'integer', 'description': 'Results (default 5)', 'default': 5}
                },
                'required': ['query']
            }
        }
    },
    {
        'type': 'function',
        'function': {
            'name': 'fetch_url',
            'description': (
                'Fetch a URL, strip HTML, index the text. '
                'Use after web_search to read a full page.'
            ),
            'parameters': {
                'type': 'object',
                'properties': {
                    'url':       {'type': 'string',  'description': 'Full URL to fetch'},
                    'max_chars': {'type': 'integer', 'description': 'Max chars to extract (default 3000)', 'default': 3000}
                },
                'required': ['url']
            }
        }
    },
    {
        'type': 'function',
        'function': {
            'name': 'vector_search',
            'description': (
                'Semantic search over ALL indexed content '
                '(PubMed, arXiv, Semantic Scholar, web, local files). '
                'Call this AFTER indexing to retrieve targeted passages before answering.'
            ),
            'parameters': {
                'type': 'object',
                'properties': {
                    'query':         {'type': 'string',  'description': 'Semantic search query'},
                    'k':             {'type': 'integer', 'description': 'Results (default 8)', 'default': 8},
                    'filter_source': {
                        'type': 'string',
                        'description': "Filter: 'fulltext'|'abstract'|'arxiv'|'semantic_scholar'|'web'|'pdf'|'pptx'|'image'|'' (all)",
                        'default': ''
                    }
                },
                'required': ['query']
            }
        }
    },
    {
        'type': 'function',
        'function': {
            'name': 'crossref_lookup',
            'description': 'Look up a DOI via CrossRef to get journal, authors, year, and citation count.',
            'parameters': {
                'type': 'object',
                'properties': {
                    'doi': {'type': 'string', 'description': "DOI, e.g. '10.1021/jctc.9b00940'"}
                },
                'required': ['doi']
            }
        }
    },
    # ── Document loading ──────────────────────────────────────────────────────
    {
        'type': 'function',
        'function': {
            'name': 'load_local_file',
            'description': (
                'Load a local file and index its content into the vector DB. '
                'Supported formats:\n'
                '  .pdf  — pdfplumber extracts text + tables page by page\n'
                '  .pptx — python-pptx extracts slide text + speaker notes\n'
                '  .html — BeautifulSoup parses the local HTML file\n'
                '  .png/.jpg/.jpeg/.webp — GPT-4o vision describes the image\n'
                'After loading, use vector_search to query the file content.'
            ),
            'parameters': {
                'type': 'object',
                'properties': {
                    'file_path': {
                        'type': 'string',
                        'description': 'Absolute or ~ path to the file, e.g. ~/Downloads/paper.pdf'
                    }
                },
                'required': ['file_path']
            }
        }
    },
    # ── Knowledge graph ───────────────────────────────────────────────────────
    {
        'type': 'function',
        'function': {
            'name': 'kg_extract',
            'description': (
                'Extract named entities and typed relationships from a text passage '
                'and add them to the knowledge graph. Call this on important paper '
                'passages after indexing to build the KG progressively. '
                'Entity types: Method, Drug, Protein, Paper, Author, Dataset, Concept, Disease.'
            ),
            'parameters': {
                'type': 'object',
                'properties': {
                    'text':      {'type': 'string', 'description': 'Text to extract entities and relations from (max ~2500 chars)'},
                    'source_id': {'type': 'string', 'description': 'Optional label for provenance, e.g. PMID or filename', 'default': ''}
                },
                'required': ['text']
            }
        }
    },
    {
        'type': 'function',
        'function': {
            'name': 'kg_search',
            'description': (
                'Query the knowledge graph: find all entities and relationships '
                'reachable from a named entity within a given number of hops. '
                'Useful for "what is related to X?" reasoning.'
            ),
            'parameters': {
                'type': 'object',
                'properties': {
                    'entity': {'type': 'string',  'description': 'Entity name to search from (partial match supported)'},
                    'depth':  {'type': 'integer', 'description': 'Max hops from the entity (default 2)', 'default': 2}
                },
                'required': ['entity']
            }
        }
    },
    {
        'type': 'function',
        'function': {
            'name': 'kg_visualize',
            'description': (
                'Visualise the full knowledge graph inline in the notebook. '
                'backend="matplotlib" for a static colour-coded plot (default); '
                'backend="pyvis" for an interactive drag-and-zoom HTML graph.'
            ),
            'parameters': {
                'type': 'object',
                'properties': {
                    'backend': {
                        'type': 'string',
                        'description': "'matplotlib' (default) or 'pyvis'",
                        'default': 'matplotlib'
                    }
                },
                'required': []
            }
        }
    },
]

print(f'{len(TOOLS)} tools registered:')
for t in TOOLS:
    print(f'  • {t["function"]["name"]}')

## Step 3 — The Agentic Loop

Each iteration: call the model → if it requests tools, execute them and feed results back → repeat until `finish_reason == 'stop'`.

In [ ]:
SYSTEM_PROMPT = """You are an expert scientific literature research agent with access \
to PubMed, arXiv, Semantic Scholar, CrossRef, and the open web.

Your goal: answer scientific questions thoroughly and accurately by:
1. Searching multiple databases to discover relevant papers
2. Fetching and fully indexing the most promising papers
3. Running targeted vector_search queries to extract specific passages
4. Synthesizing a well-cited answer grounded in retrieved evidence

Strategy:
- Start with pubmed_search and/or semantic_scholar_search to discover papers
- Use arxiv_search for recent ML/AI/computational methods
- Fetch 3-5 of the most relevant papers with fetch_and_index_paper
- Run vector_search with targeted sub-queries before writing your answer
- Use crossref_lookup to verify journal and citation counts for key papers
- For non-academic topics, use web_search + fetch_url

Citation format:
- PubMed papers: [PMID:XXXXX] or (Author et al., YEAR, PMID:XXXXX)
- arXiv preprints: [arXiv:XXXX.XXXXX]
- Web sources: [URL]

Be thorough but efficient. Always ground your answer in retrieved content."""


def run_agent(
    question: str,
    model: str = 'gpt-4o',
    max_iterations: int = 15,
    verbose: bool = True
) -> str:
    messages  = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': question},
    ]
    iteration = 0

    if verbose:
        print('=' * 65)
        print(f'QUESTION: {question}')
        print('=' * 65)

    while iteration < max_iterations:
        iteration += 1
        if verbose:
            print(f'\n[Iter {iteration}] Calling {model}...')

        response = client.chat.completions.create(
            model=model,
            messages=messages,
            tools=TOOLS,
            tool_choice='auto'
        )

        choice  = response.choices[0]
        message = choice.message
        messages.append(message)

        if choice.finish_reason == 'stop':
            final = message.content or ''
            if verbose:
                print('\n' + '=' * 65)
                print('FINAL ANSWER:')
                print(final)
                print('=' * 65)
            return final

        if choice.finish_reason != 'tool_calls':
            partial = message.content or ''
            if verbose:
                print(f'\n[finish_reason: {choice.finish_reason}]')
            return partial or f'Stopped: {choice.finish_reason}'

        for tool_call in message.tool_calls:
            tool_name = tool_call.function.name
            tool_args = json.loads(tool_call.function.arguments)

            if verbose:
                print(f'  [Tool] {tool_name}({json.dumps(tool_args)[:120]})')

            if tool_name in TOOL_REGISTRY:
                try:
                    result = TOOL_REGISTRY[tool_name](**tool_args)
                except Exception as exc:
                    result = {'error': str(exc), 'tool': tool_name}
            else:
                result = {'error': f'Unknown tool: {tool_name}'}

            result_str = json.dumps(result, ensure_ascii=False)
            if verbose:
                print(f'       → {result_str[:250]}{"..." if len(result_str) > 250 else ""}')

            messages.append({
                'role':         'tool',
                'tool_call_id': tool_call.id,
                'content':      result_str,
            })

    return 'Agent reached max_iterations without a final answer.'


print('run_agent() ready.')

## Step 4 — Demo Queries

In [ ]:
answer1 = run_agent(
    'What is the SILCS method, how does it compute binding affinity using FragMaps, '
    'and what are its key advantages over traditional FEP calculations?'
)

In [ ]:
answer2 = run_agent(
    'Compare classical docking methods (AutoDock, Glide) with deep learning '
    'approaches (DiffDock, EquiBind) for protein-ligand binding prediction. '
    'What are the accuracy tradeoffs and computational costs?'
)

In [ ]:
answer3 = run_agent(
    'Based on SILCS and modern ML docking methods, what is the most promising '
    'strategy for lead optimisation when both accuracy and throughput are required?'
)

## Step 5 — Knowledge Graph

`kg_extract` works on **any text already in the vector DB** — PubMed full-text, arXiv abstracts, Semantic Scholar results, web pages, and local files all feed the same graph.

| Tool | Input | Output |
|------|-------|--------|
| `kg_extract(text, source_id)` | Any indexed passage | Entities + relations → KG nodes/edges |
| `kg_search(entity, depth)` | Entity name | Neighbours up to N hops |
| `kg_visualize(backend)` | — | `matplotlib` static plot or `pyvis` interactive HTML |
| `load_local_file(path)` | PDF / PPTX / HTML / image | Text indexed into vector DB (then feed to `kg_extract`) |

In [ ]:
# ── Option A: build KG from online search results (runs after the demo cells) ─
hits = tool_vector_search('protein ligand binding drug discovery', k=12)
for hit in hits['results']:
    tool_kg_extract(
        hit['content'],
        source_id=hit['metadata'].get('pmid') or hit['metadata'].get('title', '?')
    )

# ── Option B: load a local file first, then add it to the KG ─────────────────
# Supported: .pdf  .pptx  .html  .png  .jpg  .webp
#
# FILE_PATH = '~/Downloads/your_paper.pdf'           # ← change this
# tool_load_local_file(FILE_PATH)
# hits = tool_vector_search('your topic', k=8, filter_source='pdf')
# for hit in hits['results']:
#     tool_kg_extract(hit['content'], source_id=pathlib.Path(FILE_PATH).name)

# ── Visualise ─────────────────────────────────────────────────────────────────
tool_kg_visualize(backend='matplotlib')   # static, colour-coded by entity type
# tool_kg_visualize(backend='pyvis')      # interactive HTML — drag, zoom, hover

# ── Query a specific entity ───────────────────────────────────────────────────
print(tool_kg_search('SILCS', depth=2))

In [ ]:
def run_conversation(questions: list[str], model: str = 'gpt-4o') -> None:
    """Multi-turn conversation — shared message history across turns."""
    messages = [{'role': 'system', 'content': SYSTEM_PROMPT}]

    for i, question in enumerate(questions):
        print(f'\n{"="*65}\nTURN {i+1}: {question}\n{"="*65}')
        messages.append({'role': 'user', 'content': question})

        for _ in range(12):
            response = client.chat.completions.create(
                model=model, messages=messages, tools=TOOLS, tool_choice='auto'
            )
            choice  = response.choices[0]
            message = choice.message
            messages.append(message)

            if choice.finish_reason == 'stop':
                print(f'\nANSWER: {message.content}')
                break

            if choice.finish_reason != 'tool_calls':
                break

            for tc in message.tool_calls:
                args       = json.loads(tc.function.arguments)
                result     = TOOL_REGISTRY.get(tc.function.name, lambda **_: {'error': 'unknown'})(**args)
                result_str = json.dumps(result)
                print(f'  [Tool] {tc.function.name}({json.dumps(args)[:80]})')
                print(f'       → {result_str[:200]}')
                messages.append({'role': 'tool', 'tool_call_id': tc.id, 'content': result_str})


run_conversation([
    'What is the SILCS-MC method for ligand binding prediction?',
    'How many papers have applied this to hERG cardiotoxicity? Give me key findings.',
    'What would be the next logical research direction based on these findings?'
])

## Step 6 — Gradio Chat UI

A two-panel interface: **chat** on the left (with file upload), **tool activity log + knowledge graph** on the right.

- Type a question and press **Send** or hit Enter
- Upload a **PDF / PPTX / HTML / image** to add it to the vector DB before asking
- Click **Refresh knowledge graph** to render the KG built so far
- **Clear chat** resets the conversation (the vector DB and KG stay populated)

In [ ]:
import gradio as gr
import copy

# ── Helpers ────────────────────────────────────────────────────────────────────

def _msg_to_dict(msg) -> dict:
    """Convert a ChatCompletionMessage object to a plain dict for gr.State storage."""
    if isinstance(msg, dict):
        return msg
    d = {'role': msg.role, 'content': msg.content}
    if getattr(msg, 'tool_calls', None):
        d['tool_calls'] = [
            {
                'id': tc.id,
                'type': 'function',
                'function': {'name': tc.function.name, 'arguments': tc.function.arguments},
            }
            for tc in msg.tool_calls
        ]
    return d


def _stream_agent(user_msg: str, messages: list, activity: str,
                  model: str = 'gpt-4o', max_iterations: int = 12):
    """Generator: yields (messages, activity_log, final_answer_or_None) each step."""
    messages = list(messages)
    messages.append({'role': 'user', 'content': user_msg})
    lines = [activity.strip()] if activity.strip() else []

    for iteration in range(1, max_iterations + 1):
        lines.append(f'[Iter {iteration}] Calling model...')
        yield messages, '\n'.join(lines), None

        response = client.chat.completions.create(
            model=model, messages=messages, tools=TOOLS, tool_choice='auto'
        )
        choice  = response.choices[0]
        message = choice.message
        messages.append(_msg_to_dict(message))

        if choice.finish_reason == 'stop':
            yield messages, '\n'.join(lines), message.content or ''
            return

        if choice.finish_reason != 'tool_calls':
            yield messages, '\n'.join(lines), message.content or f'[Stopped: {choice.finish_reason}]'
            return

        for tc in message.tool_calls:
            tool_name = tc.function.name
            tool_args = json.loads(tc.function.arguments)
            lines.append(f'  -> {tool_name}({json.dumps(tool_args)[:100]})')
            yield messages, '\n'.join(lines), None

            if tool_name in TOOL_REGISTRY:
                try:
                    result = TOOL_REGISTRY[tool_name](**tool_args)
                except Exception as exc:
                    result = {'error': str(exc)}
            else:
                result = {'error': f'Unknown tool: {tool_name}'}

            result_str = json.dumps(result, ensure_ascii=False)
            lines.append(f'     ok {result_str[:180]}')
            messages.append({'role': 'tool', 'tool_call_id': tc.id, 'content': result_str})
            yield messages, '\n'.join(lines), None

    yield messages, '\n'.join(lines), 'Max iterations reached without a final answer.'


def _kg_figure():
    """Return a matplotlib Figure of the current knowledge graph."""
    fig, ax = plt.subplots(figsize=(13, 9))
    if KG.number_of_nodes() == 0:
        ax.text(0.5, 0.5, 'Knowledge graph is empty.\nAsk a question first.',
                ha='center', va='center', fontsize=13, color='grey',
                transform=ax.transAxes)
        ax.axis('off')
        return fig

    pos         = nx.spring_layout(KG, k=2.5, seed=42)
    node_colors = [KG_TYPE_COLORS.get(KG.nodes[n].get('type', 'Unknown'), '#9E9E9E')
                   for n in KG.nodes()]
    node_sizes  = [500 + 35 * KG.degree(n) for n in KG.nodes()]

    nx.draw_networkx_nodes(KG, pos, node_color=node_colors,
                           node_size=node_sizes, alpha=0.9, ax=ax)
    nx.draw_networkx_labels(KG, pos, font_size=7, font_weight='bold', ax=ax)
    nx.draw_networkx_edges(KG, pos, edge_color='#888', arrows=True,
                           arrowsize=14, connectionstyle='arc3,rad=0.1', ax=ax)
    edge_labels = {(u, v): d.get('relation', '') for u, v, d in KG.edges(data=True)}
    nx.draw_networkx_edge_labels(KG, pos, edge_labels, font_size=6, ax=ax)

    present_types = {KG.nodes[n].get('type', 'Unknown') for n in KG.nodes()}
    legend = [mpatches.Patch(color=c, label=t)
              for t, c in KG_TYPE_COLORS.items() if t in present_types]
    ax.legend(handles=legend, loc='upper left', fontsize=8)
    ax.set_title(f'Knowledge Graph — {KG.number_of_nodes()} nodes, '
                 f'{KG.number_of_edges()} edges', fontsize=13)
    ax.axis('off')
    plt.tight_layout()
    return fig


# ── Gradio app ─────────────────────────────────────────────────────────────────

_INIT_MESSAGES = [{'role': 'system', 'content': SYSTEM_PROMPT}]

with gr.Blocks(title='Agentic RAG', theme=gr.themes.Soft()) as app:

    gr.Markdown(
        '# Agentic RAG — Scientific Research Assistant\n'
        'Ask any scientific question. The agent will autonomously search '
        'PubMed, arXiv, Semantic Scholar, and the web, then synthesise a '
        'cited answer grounded in real papers.'
    )

    msg_state = gr.State(value=copy.deepcopy(_INIT_MESSAGES))

    with gr.Row():

        # ── Left column: chat ─────────────────────────────────────────────────
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(
                label='Conversation',
                height=460,
                show_copy_button=True,
                bubble_full_width=False,
            )
            user_input = gr.Textbox(
                placeholder='Ask a scientific question and press Enter…',
                label='Your question',
                lines=2,
            )
            with gr.Row():
                send_btn  = gr.Button('Send',       variant='primary',    scale=2)
                clear_btn = gr.Button('Clear chat', variant='secondary',  scale=1)

            file_upload = gr.File(
                label='Upload a local file to add to the knowledge base  '
                      '(PDF / PPTX / HTML / PNG / JPG / WEBP)',
                file_types=['.pdf', '.pptx', '.ppt',
                            '.html', '.htm',
                            '.png', '.jpg', '.jpeg', '.webp'],
                file_count='single',
            )
            upload_status = gr.Textbox(label='Upload status', interactive=False, lines=1)

        # ── Right column: activity log + KG ──────────────────────────────────
        with gr.Column(scale=2):
            activity_box = gr.Textbox(
                label='Tool activity log',
                lines=15,
                max_lines=15,
                interactive=False,
            )
            kg_btn  = gr.Button('Refresh knowledge graph', variant='secondary')
            kg_plot = gr.Plot(label='Knowledge Graph')

    # ── Event handlers ─────────────────────────────────────────────────────────

    def handle_upload(file):
        if file is None:
            return '(no file selected)'
        file_path = file if isinstance(file, str) else file.name
        result    = tool_load_local_file(file_path)
        if 'error' in result:
            return f'Error: {result["error"]}'
        return (
            f'Loaded {result["file"]}  ({result["type"]}) — '
            f'{result["chunks_indexed"]} chunks indexed  |  '
            f'DB total: {result["total_in_db"]}'
        )

    def respond(user_msg, chat_history, messages, activity):
        if not user_msg.strip():
            yield '', chat_history, messages, activity
            return

        chat_history = list(chat_history) + [[user_msg, None]]
        yield '', chat_history, messages, activity

        final_answer = None
        for new_messages, new_activity, maybe_final in _stream_agent(
            user_msg, messages, activity
        ):
            messages    = new_messages
            activity    = new_activity
            if maybe_final is not None:
                final_answer = maybe_final
            chat_history[-1][1] = final_answer if final_answer else '…'
            yield '', chat_history, messages, activity

        if final_answer is not None:
            chat_history[-1][1] = final_answer
        yield '', chat_history, messages, activity

    def clear_chat():
        return [], copy.deepcopy(_INIT_MESSAGES), ''

    # Wire up events
    send_btn.click(
        respond,
        inputs=[user_input, chatbot, msg_state, activity_box],
        outputs=[user_input, chatbot, msg_state, activity_box],
    )
    user_input.submit(
        respond,
        inputs=[user_input, chatbot, msg_state, activity_box],
        outputs=[user_input, chatbot, msg_state, activity_box],
    )
    file_upload.upload(
        handle_upload,
        inputs=[file_upload],
        outputs=[upload_status],
    )
    clear_btn.click(
        clear_chat,
        outputs=[chatbot, msg_state, activity_box],
    )
    kg_btn.click(_kg_figure, outputs=[kg_plot])

app.launch(share=False)


## Key Takeaways

### What makes this agentic
- **Tool selection** — the model decides which databases to query per question
- **Adaptive retrieval** — full text fetched only for high-value papers
- **Progressive knowledge graph** — entities and relations accumulate across tool calls
- **Persistent vector DB** — all indexed content reused across turns; later questions are faster

### Tool inventory
| # | Tool | Source | Auto-indexes |
|---|------|--------|:---:|
| 1 | `pubmed_search` | NCBI E-utilities | — |
| 2 | `fetch_and_index_paper` | PMC full-text | ✓ |
| 3 | `arxiv_search` | arXiv API | ✓ |
| 4 | `semantic_scholar_search` | S2 Graph API | ✓ |
| 5 | `web_search` | DuckDuckGo | — |
| 6 | `fetch_url` | Any URL | ✓ |
| 7 | `vector_search` | ChromaDB | — |
| 8 | `crossref_lookup` | CrossRef API | — |
| 9 | `load_local_file` | PDF / PPTX / HTML / image | ✓ |
| 10 | `kg_extract` | Any text → NetworkX | — |
| 11 | `kg_search` | NetworkX traversal | — |
| 12 | `kg_visualize` | matplotlib / pyvis | — |

### Production improvements
- Swap `EphemeralClient` for `PersistentClient` to keep the vector DB across sessions
- Add a cross-encoder re-ranker on `vector_search` results (`cross-encoder/ms-marco-MiniLM-L-6-v2`)
- Use `text-embedding-3-large` instead of `all-MiniLM-L6-v2` for better recall
- Cache `pubmed_search` results by query string to avoid redundant API calls